# Vigía Atoyac en Colab · demostración con túnel de Cloudflare

Este cuaderno descarga la plataforma desde GitHub, la enciende en esta máquina de Colab y la publica con un
túnel de Cloudflare. Usa **Entorno de ejecución → Ejecutar todo** (Ctrl+F9) y espera unos dos minutos: al final
aparecen la dirección pública y las cuentas de prueba, cada una con su contraseña.

**Antes de la presentación**
- Deja esta pestaña abierta. Si Colab se desconecta (por inactividad, o a las 12 h como máximo en la versión
  gratuita), la plataforma se apaga y hay que volver a ejecutar todo.
- Cada ejecución en una máquina nueva cambia la dirección `https://….trycloudflare.com` y las contraseñas, y los
  datos de la demo empiezan de nuevo (90 días simulados).
- **Dirección fija (opcional):** en Cloudflare Zero Trust (*Networks → Tunnels*) crea un túnel con nombre, agrega
  un *public hostname* que apunte a `http://localhost:8000` y guarda el token del túnel en los **Secretos** de
  Colab (icono de llave, a la izquierda) con el nombre `CF_TUNNEL_TOKEN`. Si existe, este cuaderno lo usa.
- La dirección es pública: compártela solo con el jurado y el equipo. No guardes este cuaderno en GitHub con
  las salidas, porque ahí aparecen las contraseñas.
- Plan B en la laptop: `iniciar.bat` y después `compartir.bat` hacen lo mismo con el mismo tipo de túnel.

In [ ]:
# 1) Código y dependencias
REPO = "https://github.com/Maruchan35/EcoRedPuebla.git"
RAMA = "demo-hackatec"          # cámbienla a "main" cuando fusionen esta rama
PUERTO = 8000

import os, subprocess, sys
CARPETA = "/content/EcoRedPuebla"
PLATAFORMA = CARPETA + "/plataforma"
DATOS = PLATAFORMA + "/server/datos"

if os.path.isdir(CARPETA):
    subprocess.run(["git", "-C", CARPETA, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, REPO, CARPETA], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", PLATAFORMA + "/server/requirements.txt"], check=True)
print("Código y dependencias listos.")

In [ ]:
# 2) Servidor (la primera vez genera 90 días de datos de demostración: tarda alrededor de un minuto)
import time, urllib.request

def responde(url):
    try:
        urllib.request.urlopen(url + "/api/metricas", timeout=5)
        return True
    except Exception:
        return False

if "servidor" in globals() and servidor.poll() is None and responde(f"http://127.0.0.1:{PUERTO}"):
    print(f"El servidor ya estaba encendido en el puerto {PUERTO}.")
else:
    if os.path.exists(DATOS + "/puerto.txt"):
        os.remove(DATOS + "/puerto.txt")
    entorno = dict(os.environ, VA_HOST="127.0.0.1", VA_PORT=str(PUERTO), PYTHONUNBUFFERED="1")
    servidor = subprocess.Popen([sys.executable, "-m", "server"], cwd=PLATAFORMA, env=entorno,
                                stdout=open("/content/servidor.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(300):
        if servidor.poll() is not None:
            print(open("/content/servidor.log", encoding="utf-8", errors="replace").read()[-3000:])
            raise RuntimeError("El servidor se detuvo al arrancar; arriba está su registro.")
        if os.path.exists(DATOS + "/puerto.txt"):
            # Si el puerto pedido estaba ocupado, el servidor toma el siguiente libre y lo anota aquí
            PUERTO = int(open(DATOS + "/puerto.txt").read().strip() or PUERTO)
            if responde(f"http://127.0.0.1:{PUERTO}"):
                break
        time.sleep(1)
    else:
        raise RuntimeError("El servidor no respondió en 5 minutos; revisa /content/servidor.log")
    print(f"Servidor listo en el puerto {PUERTO}.")

In [ ]:
# 3) Túnel de Cloudflare: descarga cloudflared y comprueba su huella SHA-256 (igual que compartir.bat)
import hashlib, re
VERSION_CF = "2026.10.7"
HUELLA_CF = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"   # cloudflared-linux-amd64 oficial
CF = "/content/cloudflared"

if not os.path.exists(CF):
    subprocess.run(["curl", "-sSL", "--fail", "-o", CF + ".descarga",
                    f"https://github.com/cloudflare/cloudflared/releases/download/{VERSION_CF}/cloudflared-linux-amd64"], check=True)
    if hashlib.sha256(open(CF + ".descarga", "rb").read()).hexdigest() != HUELLA_CF:
        os.remove(CF + ".descarga")
        raise RuntimeError("La huella de cloudflared no coincide con la oficial: no se usa ese archivo.")
    os.rename(CF + ".descarga", CF)
    os.chmod(CF, 0o755)

token = ""
try:
    from google.colab import userdata
    token = userdata.get("CF_TUNNEL_TOKEN") or ""
except Exception:
    pass                                         # sin el secreto: túnel rápido con dirección temporal

if "tunel" in globals() and tunel.poll() is None:
    tunel.terminate()
    tunel.wait()
if token:
    tunel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "run"], env=dict(os.environ, TUNNEL_TOKEN=token),
                             stdout=open("/content/tunel.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(10)
    if tunel.poll() is not None:
        print(open("/content/tunel.log").read()[-2000:])
        raise RuntimeError("El túnel con nombre no arrancó; revisa el token en los Secretos de Colab.")
    print("Túnel con nombre encendido: la plataforma está en la dirección fija que configuraste en Cloudflare.")
else:
    tunel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PUERTO}"],
                             stdout=open("/content/tunel.log", "w"), stderr=subprocess.STDOUT)
    URL = None
    for _ in range(90):
        hallado = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", open("/content/tunel.log").read())
        if hallado or tunel.poll() is not None:
            URL = hallado.group(0) if hallado else None
            break
        time.sleep(1)
    if not URL:
        print(open("/content/tunel.log").read()[-2000:])
        raise RuntimeError("Cloudflare no entregó la dirección del túnel; vuelve a ejecutar esta celda.")
    for _ in range(30):                          # una dirección nueva tarda unos segundos en responder
        if responde(URL):
            break
        time.sleep(2)
    else:
        print("La dirección todavía no responde; espera un minuto antes de abrirla.")
    print("Plataforma (inicio de sesión):", URL + "/login.html")
    print("Vista pública:                ", URL + "/")
    print("App ciudadana:                ", URL + "/app/")

In [ ]:
# 4) Cuentas de prueba: una por rol, cada una con su contraseña (cambian en cada máquina nueva de Colab)
print(open(DATOS + "/cuentas_demo.txt", encoding="utf-8").read())

## Revisar o apagar
La celda siguiente dice si el servidor y el túnel siguen encendidos y muestra las últimas líneas del registro.
Para apagar todo: **Entorno de ejecución → Desconectar y borrar el entorno de ejecución**.

In [ ]:
print("Servidor:", "encendido" if servidor.poll() is None else f"detenido (código {servidor.returncode})")
print("Túnel:   ", "encendido" if tunel.poll() is None else f"detenido (código {tunel.returncode})")
print("\n".join(open("/content/servidor.log", encoding="utf-8", errors="replace").read().splitlines()[-15:]))